# Đề tài: Biểu diễn đặc trưng và gom cụm văn bản báo chí tiếng Việt theo nhánh lexical

## Bối cảnh
Sau Notebook 2, dữ liệu đã được tiền xử lý và lưu thành file `bai_test_do_an_preprocessed.csv` với các cột văn bản phục vụ trực tiếp cho bài toán gom cụm. Trong đó, cột `cluster_text_lexical` là đầu ra quan trọng nhất cho nhánh lexical vì đã trải qua các bước ghép văn bản, chuẩn hóa, tách từ tiếng Việt và loại bỏ stopwords phù hợp với chính corpus nghiên cứu.

## Mục tiêu
Notebook này tập trung vào việc xây dựng baseline gom cụm theo nhánh lexical, bao gồm:
- sử dụng `cluster_text_lexical` làm đầu vào chính,
- biểu diễn văn bản bằng TF-IDF và BM25,
- giảm chiều bằng TruncatedSVD,
- thực hiện gom cụm bằng K-Means,
- đánh giá kết quả bằng các chỉ số có giám sát và không giám sát,
- và so sánh các pipeline để chọn cấu hình phù hợp hơn cho bước phân tích cụm sau đó.

## Ý nghĩa học thuật
Notebook này đóng vai trò kết nối giữa bước tiền xử lý văn bản và bước phân tích cụm. Việc so sánh TF-IDF và BM25 trên cùng một đầu vào lexical giúp làm rõ ảnh hưởng của phương pháp tính trọng số từ đến độ tách biệt cụm, đồng thời tạo cơ sở thực nghiệm để lựa chọn pipeline phù hợp cho dữ liệu báo chí tiếng Việt.

## Đầu ra mong đợi
Sau notebook này, đầu ra mong đợi gồm:
- các ma trận đặc trưng văn bản theo nhánh lexical,
- các phiên bản dữ liệu đã giảm chiều,
- kết quả gom cụm từ ít nhất một pipeline baseline,
- bảng đánh giá bằng NMI, ARI, Purity, Silhouette, Davies-Bouldin Index,
- và cơ sở rõ ràng để chuyển sang bước phân tích cụm hoặc mở rộng sang các pipeline nâng cao hơn.

In [1]:
# Cell 2: import thư viện và khai báo đường dẫn dữ liệu cho Notebook 3
# Cell này chuẩn bị môi trường làm việc và xác định file đầu vào đã tiền xử lý từ Notebook 2.

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import (
    normalized_mutual_info_score,
    adjusted_rand_score,
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    confusion_matrix
)

#INPUT_PATH = Path("D:/BAI TEST/data/processed_data/bai_test_do_an_preprocessed.csv")
INPUT_PATH = Path("bai_test_do_an_preprocessed.csv")
OUTPUT_DIR = Path("processed_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("File đầu vào:", INPUT_PATH)
print("Tồn tại:", INPUT_PATH.exists())

print("\nThư mục đầu ra:", OUTPUT_DIR)

File đầu vào: bai_test_do_an_preprocessed.csv
Tồn tại: True

Thư mục đầu ra: processed_data


In [2]:
# Cell 3: đọc file dữ liệu tiền xử lý và kiểm tra nhanh đầu vào cho nhánh lexical
# Cell này giúp xác nhận file đã đúng cấu trúc và cột cluster_text_lexical sẵn sàng cho bước vector hóa.

df = pd.read_csv(INPUT_PATH)

print("Kích thước dữ liệu:", df.shape)
print("\nCác cột hiện có:")
print(df.columns.tolist())

print("\nSố giá trị thiếu theo cột:")
display(df.isna().sum().to_frame("missing_count"))

print("\nMột vài dòng dữ liệu đầu vào:")
display(df[["doc_id", "category_clean", "cluster_text_lexical"]].head(3))

Kích thước dữ liệu: (8727, 9)

Các cột hiện có:
['doc_id', 'source', 'category_clean', 'published_date', 'url', 'cluster_text', 'cluster_text_clean', 'cluster_text_segmented', 'cluster_text_lexical']

Số giá trị thiếu theo cột:


,missing_count
doc_id,0
source,0
category_clean,0
published_date,0
url,0
cluster_text,0
cluster_text_clean,2
cluster_text_segmented,2
cluster_text_lexical,0



Một vài dòng dữ liệu đầu vào:


,doc_id,category_clean,cluster_text_lexical
0,1,cong_nghe,làn_sóng máy_tính lượng_tử cơn_sốt ai quý thế_...
1,2,cong_nghe,meta đưa ai messenger trả_lời khách_hàng 24 7 ...
2,3,cong_nghe,hà_tiên sử_dụng trí_tuệ nhân_tạo phục_vụ hành_...


In [3]:
# Cell 4: làm sạch nhanh cột đầu vào chính cho nhánh lexical
# Cell này đảm bảo cluster_text_lexical không có giá trị thiếu, không có chuỗi rỗng
# và sẵn sàng cho bước TF-IDF ở cell tiếp theo.

df_work = df.copy()

df_work["cluster_text_lexical"] = (
    df_work["cluster_text_lexical"]
    .fillna("")
    .astype(str)
    .str.strip()
)

empty_lexical_count = (df_work["cluster_text_lexical"] == "").sum()

print("Kích thước dữ liệu làm việc:", df_work.shape)
print("Số dòng có cluster_text_lexical rỗng:", empty_lexical_count)

display(
    df_work[["doc_id", "category_clean", "cluster_text_lexical"]].head(3)
)

Kích thước dữ liệu làm việc: (8727, 9)
Số dòng có cluster_text_lexical rỗng: 0


,doc_id,category_clean,cluster_text_lexical
0,1,cong_nghe,làn_sóng máy_tính lượng_tử cơn_sốt ai quý thế_...
1,2,cong_nghe,meta đưa ai messenger trả_lời khách_hàng 24 7 ...
2,3,cong_nghe,hà_tiên sử_dụng trí_tuệ nhân_tạo phục_vụ hành_...


In [4]:
# Cell 5: vector hóa văn bản bằng TF-IDF cho nhánh lexical
# Cell này tạo ma trận đặc trưng baseline từ cluster_text_lexical để làm mốc so sánh với BM25 ở bước sau.

import time
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = df_work["cluster_text_lexical"].values

print("Đang huấn luyện TF-IDF...")
start_time = time.time()

tfidf_vectorizer = TfidfVectorizer(
    min_df=5,
    max_df=0.85,
    max_features=10000,
    norm="l2"
)

X_tfidf = tfidf_vectorizer.fit_transform(corpus)

elapsed_time = time.time() - start_time

print(f"Hoàn tất trong: {elapsed_time:.2f} giây")
print("Kích thước ma trận TF-IDF:", X_tfidf.shape)
print("Số lượng đặc trưng thực tế:", len(tfidf_vectorizer.get_feature_names_out()))

print("\n30 đặc trưng đầu tiên:")
print(tfidf_vectorizer.get_feature_names_out()[:30])

Đang huấn luyện TF-IDF...
Hoàn tất trong: 8.52 giây
Kích thước ma trận TF-IDF: (8727, 10000)
Số lượng đặc trưng thực tế: 10000

30 đặc trưng đầu tiên:
['000' '01' '02' '028' '03' '04' '05' '06' '07' '08' '09' '0h' '0h15' '10'
 '100' '1000' '100g' '100kg' '100km' '100m' '101' '102' '103' '104' '105'
 '106' '107' '108' '109' '10h']


In [5]:
# Cell 6: vector hóa văn bản bằng BM25 cho nhánh lexical
# Cell này tạo ma trận đặc trưng BM25 để so sánh trực tiếp với TF-IDF trên cùng corpus.

import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.base import BaseEstimator, TransformerMixin

class BM25Transformer(BaseEstimator, TransformerMixin):
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b

    def fit(self, X, y=None):
        X = sp.csr_matrix(X)
        self.n_docs_ = X.shape[0]
        self.doc_freq_ = np.bincount(X.indices, minlength=X.shape[1])
        self.idf_ = np.log((self.n_docs_ - self.doc_freq_ + 0.5) / (self.doc_freq_ + 0.5) + 1.0)
        self.avgdl_ = X.sum(axis=1).A1.mean()
        return self

    def transform(self, X):
        X = sp.csr_matrix(X, dtype=np.float64)
        doc_lengths = X.sum(axis=1).A1
        len_norm = (1 - self.b) + self.b * (doc_lengths / self.avgdl_)

        data = X.data.copy()
        indices = X.indices
        indptr = X.indptr

        for i in range(X.shape[0]):
            start, end = indptr[i], indptr[i + 1]
            if start == end:
                continue

            norm_i = len_norm[i]
            term_idx = indices[start:end]
            tf = data[start:end]

            data[start:end] = self.idf_[term_idx] * ((tf * (self.k1 + 1)) / (tf + self.k1 * norm_i))

        return sp.csr_matrix((data, indices, indptr), shape=X.shape)

print("Đang huấn luyện BM25...")
start_time = time.time()

count_vectorizer = CountVectorizer(
    min_df=5,
    max_df=0.85,
    max_features=10000
)

X_count = count_vectorizer.fit_transform(corpus)

bm25_transformer = BM25Transformer(k1=1.5, b=0.75)
X_bm25 = bm25_transformer.fit_transform(X_count)

elapsed_time = time.time() - start_time

print(f"Hoàn tất trong: {elapsed_time:.2f} giây")
print("Kích thước ma trận BM25:", X_bm25.shape)
print("Số lượng đặc trưng thực tế:", len(count_vectorizer.get_feature_names_out()))

print("\n30 đặc trưng đầu tiên:")
print(count_vectorizer.get_feature_names_out()[:30])

Đang huấn luyện BM25...
Hoàn tất trong: 2.19 giây
Kích thước ma trận BM25: (8727, 10000)
Số lượng đặc trưng thực tế: 10000

30 đặc trưng đầu tiên:
['000' '01' '02' '028' '03' '04' '05' '06' '07' '08' '09' '0h' '0h15' '10'
 '100' '1000' '100g' '100kg' '100km' '100m' '101' '102' '103' '104' '105'
 '106' '107' '108' '109' '10h']


In [7]:
# Cell 7: giảm chiều bằng TruncatedSVD cho hai ma trận TF-IDF và BM25
# Cell này đưa dữ liệu từ không gian 10.000 chiều về không gian thấp hơn,
# giúp K-Means hoạt động ổn định hơn và giảm chi phí tính toán.

import time

N_COMPONENTS = 300

print(f"Đang giảm chiều với TruncatedSVD, số chiều đích = {N_COMPONENTS} ...")

start_time = time.time()

svd_tfidf = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
X_tfidf_svd = svd_tfidf.fit_transform(X_tfidf)

svd_bm25 = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
X_bm25_svd = svd_bm25.fit_transform(X_bm25)

elapsed_time = time.time() - start_time

print(f"Hoàn tất trong: {elapsed_time:.2f} giây")

print("\nTF-IDF + SVD:")
print("Shape:", X_tfidf_svd.shape)
print("Explained variance ratio sum:", round(svd_tfidf.explained_variance_ratio_.sum(), 4))

print("\nBM25 + SVD:")
print("Shape:", X_bm25_svd.shape)
print("Explained variance ratio sum:", round(svd_bm25.explained_variance_ratio_.sum(), 4))

Đang giảm chiều với TruncatedSVD, số chiều đích = 300 ...
Hoàn tất trong: 23.07 giây

TF-IDF + SVD:
Shape: (8727, 300)
Explained variance ratio sum: 0.4469

BM25 + SVD:
Shape: (8727, 300)
Explained variance ratio sum: 0.3031


In [8]:

# Cell 8: khảo sát số cụm k bằng Elbow và Silhouette trên hai nhánh TF-IDF và BM25
# Cell này giúp chọn k hợp lý hơn trước khi chạy K-Means chính thức.

k_values = list(range(2, 13))

tfidf_inertia = []
tfidf_silhouette = []

bm25_inertia = []
bm25_silhouette = []

print("Đang khảo sát k cho nhánh TF-IDF...")
for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=10,
        random_state=42
    )
    labels = kmeans.fit_predict(X_tfidf_svd)
    tfidf_inertia.append(kmeans.inertia_)
    tfidf_silhouette.append(silhouette_score(X_tfidf_svd, labels))
    print(f"TF-IDF | k = {k}: inertia = {kmeans.inertia_:.2f}, silhouette = {tfidf_silhouette[-1]:.4f}")

print("\nĐang khảo sát k cho nhánh BM25...")
for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=10,
        random_state=42
    )
    labels = kmeans.fit_predict(X_bm25_svd)
    bm25_inertia.append(kmeans.inertia_)
    bm25_silhouette.append(silhouette_score(X_bm25_svd, labels))
    print(f"BM25   | k = {k}: inertia = {kmeans.inertia_:.2f}, silhouette = {bm25_silhouette[-1]:.4f}")

Đang khảo sát k cho nhánh TF-IDF...
TF-IDF | k = 2: inertia = 3646.09, silhouette = 0.0507
TF-IDF | k = 3: inertia = 3556.81, silhouette = 0.0535
TF-IDF | k = 4: inertia = 3481.43, silhouette = 0.0340
TF-IDF | k = 5: inertia = 3416.70, silhouette = 0.0324
TF-IDF | k = 6: inertia = 3375.12, silhouette = 0.0362
TF-IDF | k = 7: inertia = 3323.11, silhouette = 0.0300
TF-IDF | k = 8: inertia = 3292.40, silhouette = 0.0310
TF-IDF | k = 9: inertia = 3249.06, silhouette = 0.0370
TF-IDF | k = 10: inertia = 3213.25, silhouette = 0.0418
TF-IDF | k = 11: inertia = 3187.94, silhouette = 0.0426
TF-IDF | k = 12: inertia = 3161.68, silhouette = 0.0424

Đang khảo sát k cho nhánh BM25...
BM25   | k = 2: inertia = 6310765.64, silhouette = 0.0882
BM25   | k = 3: inertia = 6137126.19, silhouette = 0.0611
BM25   | k = 4: inertia = 6007543.68, silhouette = 0.0303
BM25   | k = 5: inertia = 5905502.47, silhouette = 0.0272
BM25   | k = 6: inertia = 5831816.60, silhouette = 0.0269
BM25   | k = 7: inertia = 57716

In [9]:
# Cell 9: so sánh hai nhánh TF-IDF và BM25 tại k = 6 và k = 8
# Cell này đánh giá cả theo nhãn ngoài và cấu trúc nội tại để chọn cấu hình phù hợp hơn.

true_labels = df_work["category_clean"].values
k_targets = [6, 8]
results = []

def purity_score(y_true, y_pred):
    contingency = pd.crosstab(pd.Series(y_true, name="true"), pd.Series(y_pred, name="pred"))
    return contingency.max(axis=0).sum() / contingency.values.sum()

def evaluate_model(X_data, method_name):
    for k in k_targets:
        kmeans = KMeans(
            n_clusters=k,
            init="k-means++",
            n_init=10,
            random_state=42
        )
        labels = kmeans.fit_predict(X_data)

        results.append({
            "Phuong_phap": method_name,
            "k": k,
            "n_clusters_found": len(np.unique(labels)),
            "Silhouette": round(silhouette_score(X_data, labels), 4),
            "ARI": round(adjusted_rand_score(true_labels, labels), 4),
            "NMI": round(normalized_mutual_info_score(true_labels, labels), 4),
            "Purity": round(purity_score(true_labels, labels), 4),
            "Davies_Bouldin": round(davies_bouldin_score(X_data, labels), 4),
            "Calinski_Harabasz": round(calinski_harabasz_score(X_data, labels), 4),
        })

print("Đang chạy đánh giá cho k = 6 và k = 8...")

evaluate_model(X_tfidf_svd, "TF-IDF + SVD(300)")
evaluate_model(X_bm25_svd, "BM25 + SVD(300)")

results_compare_df = pd.DataFrame(results)
display(results_compare_df.sort_values(by=["NMI", "ARI", "Purity"], ascending=False).reset_index(drop=True))

Đang chạy đánh giá cho k = 6 và k = 8...


,Phuong_phap,k,n_clusters_found,Silhouette,ARI,NMI,Purity,Davies_Bouldin,Calinski_Harabasz
0,TF-IDF + SVD(300),8,8,0.0310,0.4350,0.5973,0.7731,4.2279,175.4672
1,TF-IDF + SVD(300),6,6,0.0362,0.3930,0.5952,0.7232,4.3073,196.9391
2,BM25 + SVD(300),6,6,0.0269,0.5315,0.5862,0.7629,4.3216,208.7923
3,BM25 + SVD(300),8,8,0.0243,0.4362,0.5396,0.7409,4.0889,178.3720
